# 실습 5: 모델 레지스트리 & 서빙

**목표**: 학습한 모델을 Unity Catalog에 등록하고, Serving Endpoint로 실시간 배포합니다.

**배울 것**:
- Unity Catalog 모델 레지스트리 (버전, Alias)
- Model Serving Endpoint 생성
- REST API로 실시간 예측

**소요 시간**: ~40분

⚠️ **비용 주의**: Serving Endpoint는 실습 후 반드시 삭제하세요!

## Step 1: 모델 레지스트리 — Alias로 버전 관리

In [0]:
import mlflow
from mlflow import MlflowClient

client = MlflowClient()

CATALOG = "3dt005_databricks"
SCHEMA = "wine"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.wine_quality_model_lab"

# 등록된 모델 정보 확인
model_info = client.get_registered_model(MODEL_NAME)
print(f"모델명: {model_info.name}")

# UC 모델은 search_model_versions로 버전 조회
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
for v in versions:
    print(f"  버전 {v.version} | 상태: {v.status} | 생성: {v.creation_timestamp}")

모델명: 3dt005_databricks.wine.wine_quality_model_lab
  버전 2 | 상태: READY | 생성: 1774328961166
  버전 1 | 상태: READY | 생성: 1774318182197


In [0]:
# Feature Store로 로깅된 모델은 온라인 FS 설정 없이 서빙 불가
# → 내부의 raw sklearn 모델을 추출하여 순수 sklearn 모델로 재등록
import os, pickle, yaml
import pandas as pd
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

# 최신 버전의 모델 아티팩트 다운로드
latest_version = max(client.search_model_versions(f"name='{MODEL_NAME}'"), key=lambda v: int(v.version))
local_path = mlflow.artifacts.download_artifacts(f"models:/{MODEL_NAME}/{latest_version.version}")

# Feature Store 래퍼 여부 확인
with open(f"{local_path}/MLmodel", "r") as f:
    mlmodel = yaml.safe_load(f)

loader = mlmodel.get("flavors", {}).get("python_function", {}).get("loader_module", "")

if "feature_store" in loader:
    print("⚠️ Feature Store 모델 감지 → 순수 sklearn 모델로 재등록합니다...")
    raw_model_path = os.path.join(local_path, "data", "feature_store", "raw_model", "model.pkl")
    with open(raw_model_path, "rb") as f:
        sklearn_model = pickle.load(f)

    # 서빙용 feature columns (wine_id 제외)
    sample_input = pd.DataFrame([{
        "fixed_acidity": 7.0, "volatile_acidity": 0.27, "citric_acid": 0.36,
        "residual_sugar": 20.7, "chlorides": 0.045, "free_sulfur_dioxide": 45.0,
        "total_sulfur_dioxide": 170.0, "density": 1.001, "pH": 3.0,
        "sulphates": 0.45, "alcohol": 8.8, "free_sulfur_ratio": 0.265,
        "acidity_alcohol_interaction": 61.6, "sugar_alcohol_ratio": 2.352
    }])
    signature = infer_signature(sample_input, sklearn_model.predict(sample_input))

    with mlflow.start_run(run_name="re-log-for-serving") as run:
        mlflow.sklearn.log_model(
            sk_model=sklearn_model,
            artifact_path="model",
            signature=signature,
            input_example=sample_input,
            registered_model_name=MODEL_NAME,
        )
    new_version = max(client.search_model_versions(f"name='{MODEL_NAME}'"), key=lambda v: int(v.version))
    print(f"✅ 순수 sklearn 모델 등록 완료! 버전: {new_version.version}")
    SERVE_VERSION = new_version.version
else:
    print("✅ 이미 순수 모델입니다. 재등록 불필요.")
    SERVE_VERSION = latest_version.version

print(f"   서빙에 사용할 버전: {SERVE_VERSION}")

✅ 이미 순수 모델입니다. 재등록 불필요.
   서빙에 사용할 버전: 3


In [0]:
# @champion Alias 부여 (서빙 가능한 버전에 설정)
client.set_registered_model_alias(MODEL_NAME, "champion", SERVE_VERSION)
print(f"✅ 모델 버전 {SERVE_VERSION}에 @champion 별칭 부여 완료!")

# Alias로 모델 로드
champion_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
print(f"✅ @champion 모델 로드 성공!")

✅ 모델 버전 3에 @champion 별칭 부여 완료!


✅ @champion 모델 로드 성공!


## Step 2: Serving Endpoint 생성

In [0]:
import requests
import json

# Databricks API 토큰
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = spark.conf.get("spark.databricks.workspaceUrl")

ENDPOINT_NAME = "wine-quality-serving-lab"

# Serving Endpoint 생성 (Scale to zero 설정!)
# ⚠️ auto_capture_config(Legacy Inference Table)는 deprecated 되어 제거됨
endpoint_config = {
    "name": ENDPOINT_NAME,
    "config": {
        "served_entities": [
            {
                "entity_name": MODEL_NAME,
                "entity_version": str(SERVE_VERSION),
                "workload_size": "Small",
                "scale_to_zero_enabled": True,  # 💰 비용 절감!
            }
        ],
    },
}

response = requests.post(
    f"https://{host}/api/2.0/serving-endpoints",
    headers={"Authorization": f"Bearer {token}"},
    json=endpoint_config,
)

if response.status_code == 200:
    print(f"✅ Serving Endpoint 생성 시작: {ENDPOINT_NAME}")
    print(f"   모델 버전: {SERVE_VERSION}")
    print(f"   Scale to zero: 활성화 (요청 없으면 비용 $0)")
else:
    print(f"⚠️ 응답: {response.status_code}")
    print(response.json())

✅ Serving Endpoint 생성 시작: wine-quality-serving-lab
   모델 버전: 3
   Scale to zero: 활성화 (요청 없으면 비용 $0)


⏳ **Endpoint가 Ready 상태가 될 때까지 2~5분 소요됩니다.**

Serving UI에서 상태를 확인하세요: 왼쪽 메뉴 → Serving

In [0]:
# 엔드포인트 상태 확인
import time

for i in range(30):  # 최대 5분 대기
    status_response = requests.get(
        f"https://{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}",
        headers={"Authorization": f"Bearer {token}"},
    )

    if status_response.status_code != 200:
        print(f"⚠️ 엔드포인트 조회 실패 ({status_response.status_code}). 엔드포인트가 생성되었는지 확인하세요.")
        break

    ep_state = status_response.json().get("state", {})
    state = ep_state.get("ready", "NOT_READY")
    config_state = ep_state.get("config_update", "NOT_READY")
    print(f"[{i*10}s] Ready: {state}, Config: {config_state}")

    if state == "READY":
        print("✅ Endpoint 준비 완료!")
        break

    # 에러 상태 감지 — 무한 루프 방지
    if config_state == "UPDATE_FAILED":
        print("❌ Endpoint 설정 실패! 로그를 확인하세요.")
        print(status_response.json().get("state", {}))
        break

    time.sleep(10)
else:
    print("⚠️ 타임아웃: 5분 내에 Ready 상태에 도달하지 못했습니다. Serving UI에서 직접 확인하세요.")

[0s] Ready: READY, Config: NOT_UPDATING
✅ Endpoint 준비 완료!


## Step 3: REST API로 실시간 예측

In [0]:
# 테스트 데이터 준비
sample_data = {
    "dataframe_records": [
        {
            "fixed_acidity": 7.0,
            "volatile_acidity": 0.27,
            "citric_acid": 0.36,
            "residual_sugar": 20.7,
            "chlorides": 0.045,
            "free_sulfur_dioxide": 45.0,
            "total_sulfur_dioxide": 170.0,
            "density": 1.001,
            "pH": 3.0,
            "sulphates": 0.45,
            "alcohol": 8.8,
            "free_sulfur_ratio": 0.265,
            "acidity_alcohol_interaction": 61.6,
            "sugar_alcohol_ratio": 2.352,
        }
    ]
}

# REST API 호출
response = requests.post(
    f"https://{host}/serving-endpoints/{ENDPOINT_NAME}/invocations",
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    },
    json=sample_data,
)

if response.status_code == 200:
    prediction = response.json()
    print(f"🍷 예측 결과: {prediction}")
    pred_val = prediction.get("predictions", [None])[0]
    print(f"   품질: {'좋음 (1)' if pred_val == 1 else '보통 (0)'}")
else:
    print(f"⚠️ 에러: {response.status_code}")
    print(response.text)

🍷 예측 결과: {'predictions': [0]}
   품질: 보통 (0)


## Step 4: 실습 후 정리 (비용 관리!)

In [0]:
# ⚠️ Serving Endpoint 삭제 — 실습 후 반드시 실행!
delete_response = requests.delete(
    f"https://{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}",
    headers={"Authorization": f"Bearer {token}"},
)

if delete_response.status_code == 200:
    print(f"✅ Serving Endpoint '{ENDPOINT_NAME}' 삭제 완료!")
    print("💰 불필요한 비용 발생을 방지했습니다.")
else:
    print(f"⚠️ 삭제 실패: {delete_response.status_code}")

✅ Serving Endpoint 'wine-quality-serving-lab' 삭제 완료!
💰 불필요한 비용 발생을 방지했습니다.


## 🎯 핵심 정리

| 단계 | 방법 | 포인트 |
|------|------|--------|
| **모델 등록** | Unity Catalog 모델 레지스트리 | 버전 관리 + 거버넌스 |
| **Alias** | `@champion` / `@challenger` | 프로덕션 모델 관리 |
| **서빙** | Model Serving Endpoint | REST API 실시간 예측 |
| **비용** | Scale to zero 활성화 | 요청 없으면 $0 |

---
**다음**: 세션 6 — 추천 시스템 미니 프로젝트 🎬